In [ ]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

In [ ]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/ayush1220/cifar10"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/
!mv ./dataset/cifar10/* ./dataset/
!rm -r ./dataset/cifar10 dataset.zip

%cd ./dataset/
!zip -rq ../dataset.zip .
%cd ..

In [ ]:
# Import pip packages
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

In [ ]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [ ]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [ ]:
# Find mean, std
_dataset = datasets.ImageFolder(root='./dataset/train', transform=transforms.ToTensor())
_loader  = DataLoader(_dataset)

mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

mean, std

In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.m6i.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 50,
        'batch_size': 64,
        'lr': 0.001,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

In [ ]:
# Create Endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
)

In [ ]:
# Create new model + Endpoint Configuration
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
  name = f'model-{int(time.time())}',
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

In [ ]:
# Create/Update Endpoint
sm.create_endpoint(
    EndpointName = "<ENDPOINT_NAME>",
    EndpointConfigName = new_config
)

# sm.update_endpoint(
#     EndpointName = "<ENDPOINT_NAME>",
#     EndpointConfigName = new_config
# )

In [ ]:
# Get class list
CLASSES = _dataset.classes
CLASSES

In [ ]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'cifar100-endpoint'
image_path = "apple-1014.png"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])